In [ ]:
# CSJコーパスの他の自由対話音声にも対応可能
# CSJ音声から感情次元表現（Valence, Arousal, Dominance）を抽出してCSV保存

import os
import csv
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa
import torch
import torch.nn as nn

from transformers import Wav2Vec2Processor
from transformers.models.wav2vec2.modeling_wav2vec2 import (
    Wav2Vec2Model,
    Wav2Vec2PreTrainedModel,
)

from tqdm import tqdm


# ===== 設定 =====

# 作成した発話単位wavフォルダ
ROOT_PATH = "/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/data/Segments"

OUTPUT_DIR = "data/CSJ_VAD_results"


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SAMPLE_RATE = 16000

MODEL_NAME = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"



# ===== モデル定義 =====

class RegressionHead(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.dense = nn.Linear(
            config.hidden_size,
            config.hidden_size
        )

        self.dropout = nn.Dropout(
            config.final_dropout
        )

        self.out_proj = nn.Linear(
            config.hidden_size,
            config.num_labels
        )


    def forward(self, features, **kwargs):

        x = features

        x = self.dropout(x)

        x = self.dense(x)

        x = torch.tanh(x)

        x = self.dropout(x)

        x = self.out_proj(x)

        return x



class EmotionModel(Wav2Vec2PreTrainedModel):

    # transformers最新版対策
    _tied_weights_keys = {}
    all_tied_weights_keys = {}


    def __init__(self, config):

        super().__init__(config)

        self.config = config

        self.wav2vec2 = Wav2Vec2Model(config)

        self.classifier = RegressionHead(config)

        self.init_weights()


    def forward(self, input_values):

        outputs = self.wav2vec2(
            input_values
        )

        hidden_states = outputs[0]


        # 時間方向平均
        hidden_states = torch.mean(
            hidden_states,
            dim=1
        )


        logits = self.classifier(
            hidden_states
        )


        return hidden_states, logits



# ===== モデルロード =====

processor = Wav2Vec2Processor.from_pretrained(
    MODEL_NAME
)


model = EmotionModel.from_pretrained(
    MODEL_NAME
).to(DEVICE)


model.eval()



# ===== WAV収集 =====

def collect_wav_files(root_dir):

    # L/Rフォルダ以下も探索
    return sorted(
        Path(root_dir).rglob("*.wav")
    )



# ===== 音声読み込み =====

def load_audio(
    path,
    target_sr=SAMPLE_RATE
):

    wav, sr = sf.read(path)


    # stereo → mono
    if wav.ndim > 1:

        wav = wav.mean(axis=1)



    # sampling rate変換
    if sr != target_sr:

        wav = librosa.resample(
            y=wav,
            orig_sr=sr,
            target_sr=target_sr
        )


    return wav.astype(np.float32), target_sr



# ===== VAD推定 =====

def extract_vad(
    wav,
    sampling_rate
):

    inputs = processor(
        wav,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )


    input_values = inputs[
        "input_values"
    ].to(DEVICE)



    with torch.no_grad():

        logits = model(
            input_values
        )[1]


    logits = (
        logits
        .detach()
        .cpu()
        .numpy()[0]
    )


    # 出力順
    # [arousal, dominance, valence]

    arousal = float(logits[0])

    dominance = float(logits[1])

    valence = float(logits[2])


    return valence, arousal, dominance



# ===== メイン処理 =====

def process_csj():


    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )


    wav_files = collect_wav_files(
        ROOT_PATH
    )


    print(
        f"Found wav files: {len(wav_files)}"
    )


    results = []



    for wav_path in tqdm(
        wav_files,
        desc="Processing CSJ WAV"
    ):


        try:

            wav, sr = load_audio(
                str(wav_path)
            )


            # 0.3秒未満は除外
            if len(wav) / sr < 0.3:
                continue



            valence, arousal, dominance = extract_vad(
                wav,
                sr
            )



            # 例:
            # Segments/
            # └ D03F0001/
            #    └ L/
            #       └ 0001_L.wav

            dialog = wav_path.parent.parent.name

            speaker = wav_path.parent.name



            results.append(
                [
                    dialog,
                    speaker,
                    wav_path.name,
                    str(wav_path),
                    valence,
                    arousal,
                    dominance
                ]
            )



        except Exception as e:

            print(
                f"Error processing {wav_path.name}: {e}"
            )



    # ===== CSV保存 =====

    csv_path = os.path.join(
        OUTPUT_DIR,
        "CSJ_VAD_results.csv"
    )


    with open(
        csv_path,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:


        writer = csv.writer(f)


        writer.writerow(
            [
                "dialog",
                "speaker",
                "audio_file",
                "wav_path",
                "valence",
                "arousal",
                "dominance"
            ]
        )


        writer.writerows(
            results
        )


    print("\nSaved results:")
    print(csv_path)



# ===== 実行 =====

if __name__ == "__main__":

    process_csj()

In [ ]:
# 上記の保存先を変えたバージョン（未使用）
# CSJ音声から感情次元表現（Valence, Arousal, Dominance）を抽出してCSV保存

import os
import csv
from pathlib import Path
import numpy as np
import soundfile as sf
import librosa
import torch
import torch.nn as nn
from transformers import Wav2Vec2Processor
from transformers.models.wav2vec2.modeling_wav2vec2 import (
    Wav2Vec2Model,
    Wav2Vec2PreTrainedModel,
)

from tqdm import tqdm


# ===== 設定 =====

# 作成した発話単位wavフォルダ
ROOT_PATH = "/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/data/Segments"

OUTPUT_DIR = "data/CSJ_VAD_results"


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SAMPLE_RATE = 16000

MODEL_NAME = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"



# ===== モデル定義 =====

class RegressionHead(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.dense = nn.Linear(
            config.hidden_size,
            config.hidden_size
        )

        self.dropout = nn.Dropout(
            config.final_dropout
        )

        self.out_proj = nn.Linear(
            config.hidden_size,
            config.num_labels
        )


    def forward(self, features, **kwargs):

        x = features

        x = self.dropout(x)

        x = self.dense(x)

        x = torch.tanh(x)

        x = self.dropout(x)

        x = self.out_proj(x)

        return x



class EmotionModel(Wav2Vec2PreTrainedModel):

    # transformers最新版対策
    _tied_weights_keys = {}
    all_tied_weights_keys = {}


    def __init__(self, config):

        super().__init__(config)

        self.config = config

        self.wav2vec2 = Wav2Vec2Model(config)

        self.classifier = RegressionHead(config)

        self.init_weights()


    def forward(self, input_values):

        outputs = self.wav2vec2(
            input_values
        )

        hidden_states = outputs[0]


        # 時間方向平均
        hidden_states = torch.mean(
            hidden_states,
            dim=1
        )


        logits = self.classifier(
            hidden_states
        )


        return hidden_states, logits



# ===== モデルロード =====

processor = Wav2Vec2Processor.from_pretrained(
    MODEL_NAME
)


model = EmotionModel.from_pretrained(
    MODEL_NAME
).to(DEVICE)


model.eval()



# ===== WAV収集 =====

def collect_wav_files(root_dir):

    # L/Rフォルダ以下も探索
    return sorted(
        Path(root_dir).rglob("*.wav")
    )



# ===== 音声読み込み =====

def load_audio(
    path,
    target_sr=SAMPLE_RATE
):

    wav, sr = sf.read(path)


    # stereo → mono
    if wav.ndim > 1:

        wav = wav.mean(axis=1)



    # sampling rate変換
    if sr != target_sr:

        wav = librosa.resample(
            y=wav,
            orig_sr=sr,
            target_sr=target_sr
        )


    return wav.astype(np.float32), target_sr



# ===== VAD推定 =====

def extract_vad(
    wav,
    sampling_rate
):

    inputs = processor(
        wav,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )


    input_values = inputs[
        "input_values"
    ].to(DEVICE)



    with torch.no_grad():

        logits = model(
            input_values
        )[1]


    logits = (
        logits
        .detach()
        .cpu()
        .numpy()[0]
    )


    # 出力順
    # [arousal, dominance, valence]

    arousal = float(logits[0])

    dominance = float(logits[1])

    valence = float(logits[2])


    return valence, arousal, dominance



# ===== メイン処理 =====

def process_csj():


    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )


    wav_files = collect_wav_files(
        ROOT_PATH
    )


    print(
        f"Found wav files: {len(wav_files)}"
    )


    results = []



    for wav_path in tqdm(
        wav_files,
        desc="Processing CSJ WAV"
    ):


        try:

            wav, sr = load_audio(
                str(wav_path)
            )


            # 0.3秒未満は除外
            if len(wav) / sr < 0.3:
                continue



            valence, arousal, dominance = extract_vad(
                wav,
                sr
            )



            # 例:
            # Segments/
            # └ D03F0001/
            #    └ L/
            #       └ 0001_L.wav

            dialog = wav_path.parent.parent.name

            speaker = wav_path.parent.name



            results.append(
                [
                    dialog,
                    speaker,
                    wav_path.name,
                    str(wav_path),
                    valence,
                    arousal,
                    dominance
                ]
            )



        except Exception as e:

            print(
                f"Error processing {wav_path.name}: {e}"
            )



    # ===== CSV保存 =====

def process_csj():

    wav_files = collect_wav_files(ROOT_PATH)

    print(
        f"Found wav files: {len(wav_files)}"
    )


    # 対話ごとに保存するための辞書
    vad_results = {}


    for wav_path in tqdm(
        wav_files,
        desc="Processing CSJ WAV"
    ):

        try:

            wav, sr = load_audio(
                str(wav_path)
            )


            if len(wav) / sr < 0.3:
                continue


            valence, arousal, dominance = extract_vad(
                wav,
                sr
            )


            # D03F0001
            dialog = wav_path.parent.parent.name

            # L/R
            speaker = wav_path.parent.name



            if dialog not in vad_results:
                vad_results[dialog] = []


            vad_results[dialog].append(
                [
                    speaker,
                    wav_path.name,
                    valence,
                    arousal,
                    dominance
                ]
            )


        except Exception as e:

            print(
                f"Error processing {wav_path.name}: {e}"
            )

    # ==========================
    # 対話ごとにCSV保存
    # ==========================

    for dialog, results in vad_results.items():

        save_dir = os.path.join(
            ROOT_PATH,
            dialog
        )


        csv_path = os.path.join(
            save_dir,
            f"{dialog}_VAD.csv"
        )


        with open(
            csv_path,
            "w",
            newline="",
            encoding="utf-8"
        ) as f:


            writer = csv.writer(f)


            writer.writerow(
                [
                    "speaker",
                    "audio_file",
                    "valence",
                    "arousal",
                    "dominance"
                ]
            )


            writer.writerows(
                results
            )


        print(
            f"Saved: {csv_path}"
        )



# ===== 実行 =====

if __name__ == "__main__":

    process_csj()

Loading weights: 100%|██████████| 234/234 [00:00<00:00, 3592.34it/s]
[W720 22:39:10.685912441 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 16777216 bytes (free: 9437184, total: 10354753536).
[W720 22:39:10.697250505 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 20971520 bytes (free: 3145728, total: 10354753536).
[W720 22:39:10.697528341 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 20971520 bytes (free: 3145728, total: 10354753536).


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 9.64 GiB of which 3.00 MiB is free. Process 974757 has 8.26 GiB memory in use. Including non-PyTorch memory, this process has 1.37 GiB memory in use. Of the allocated memory 1.10 GiB is allocated by PyTorch, and 8.62 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)